In [2]:
import pandas as pd
import numpy as np
import os,sys

In [3]:
# Load the historical flights dataset
print("Loading historical flights dataset...")
data_path = "/work/kristsey/avinor/data/historical_flights.csv"

# Load data
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

Loading historical flights dataset...
Dataset shape: (410442, 11)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410442 entries, 0 to 410441
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          410442 non-null  object
 1   dep_airport        410442 non-null  object
 2   dep_airport_group  232968 non-null  object
 3   arr_airport        410442 non-null  object
 4   arr_airport_group  231669 non-null  object
 5   service_type       410336 non-null  object
 6   std                410442 non-null  object
 7   sta                410442 non-null  object
 8   cancelled          410442 non-null  int64 
 9   atd                394327 non-null  object
 10  ata                393576 non-null  object
dtypes: int64(1), object(10)
memory usage: 34.4+ MB
None

First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV            

In [4]:
## Process data
# Add the parent directory to the Python path so we can import from src
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath("avinorhackathon")))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(sys.path)

# Now we can import from src
from src.preprocessing import DataPreprocessor

data_preprocessor = DataPreprocessor()
print("Processing data...")
processed_df = data_preprocessor.process_hist_flights(df)
print(f"Processed dataset shape: {processed_df.shape}")

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/kristsey/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/home/kristsey/AvinorHackathon']
Processing data...
Missing values after processing:
flight_id                 0
dep_airport               0
dep_airport_group    169367
arr_airport               0
arr_airport_group    169919
service_type              0
std                       0
sta                       0
atd                       0
ata                       0
dtype: int64
Processed dataset shape: (390856, 10)


In [8]:
sched_data = processed_df[["std","sta", "dep_airport", "arr_airport", "flight_id"]].copy()

In [14]:
sched_grouped_dep = sched_data.sort_values("std").groupby("dep_airport")
sched_grouped_arr = sched_data.sort_values("sta").groupby("arr_airport")

# create a new dataframe with only a single time feature, merging dep and arr times on matching airports, keeping all rows
sched_merged = pd.merge(sched_grouped_dep.head(1), sched_grouped_arr.head(1), left_on="std", right_on="sta", suffixes=("_dep", "_arr"), how="inner")
print(f"Merged schedule dataset shape: {sched_merged.shape}")
print("\nFirst few rows of merged schedule data:")
print(sched_merged.head())

Merged schedule dataset shape: (3, 10)

First few rows of merged schedule data:
              std_dep             sta_dep dep_airport_dep arr_airport_dep  \
0 2018-01-01 10:05:00 2018-01-01 10:35:00             SOJ             HFT   
1 2018-01-01 11:25:00 2018-01-01 11:50:00             VAW             BJF   
2 2018-01-01 11:40:00 2018-01-01 11:58:00             MEH             BVG   

  flight_id_dep             std_arr             sta_arr dep_airport_arr  \
0         WF974 2018-01-01 08:45:00 2018-01-01 10:05:00             OSY   
1         WF973 2018-01-01 10:50:00 2018-01-01 11:25:00             HFT   
2         WF974 2018-01-01 09:55:00 2018-01-01 11:40:00             OSL   

  arr_airport_arr flight_id_arr  
0             OSL         WF781  
1             MEH         WF974  
2             SSJ         WF720  


In [ ]:
# Create departure and arrival airport congestion features
print("Creating congestion features...")

# Convert time columns to datetime if they aren't already
sched_data['std'] = pd.to_datetime(sched_data['std'])
sched_data['sta'] = pd.to_datetime(sched_data['sta'])

def create_airport_congestion_features_fast(df, backward_minutes=45, forward_minutes=15):
    """
    Create airport congestion features using optimized pandas operations
    """
    print("Preparing data...")
    
    # Create separate departure and arrival datasets
    departures = df[['flight_id', 'std', 'dep_airport']].copy()
    departures.columns = ['flight_id', 'time', 'airport']
    departures['flight_type'] = 'departure'
    
    arrivals = df[['flight_id', 'sta', 'arr_airport']].copy()
    arrivals.columns = ['flight_id', 'time', 'airport']
    arrivals['flight_type'] = 'arrival'
    
    # Combine all airport activity
    all_flights = pd.concat([departures, arrivals], ignore_index=True)
    all_flights = all_flights.sort_values(['airport', 'time']).reset_index(drop=True)
    
    print(f"Processing {len(all_flights)} flight events across {all_flights['airport'].nunique()} airports...")
    
    # Use a simpler but faster approach
    results = []
    
    # Group by airport for efficiency
    grouped = all_flights.groupby('airport')
    
    for airport, airport_flights in grouped:
        airport_flights = airport_flights.sort_values('time').reset_index(drop=True)
        
        # Convert to numpy for faster operations
        times = airport_flights['time'].values
        flight_types = airport_flights['flight_type'].values
        flight_ids = airport_flights['flight_id'].values
        
        # Calculate time windows
        backward_delta = pd.Timedelta(minutes=backward_minutes)
        forward_delta = pd.Timedelta(minutes=forward_minutes)
        
        for i, (current_time, flight_id, flight_type) in enumerate(zip(times, flight_ids, flight_types)):
            window_start = current_time - backward_delta
            window_end = current_time + forward_delta
            
            # Use boolean indexing for fast filtering
            in_window_mask = (times >= window_start) & (times <= window_end)
            window_flight_types = flight_types[in_window_mask]
            
            # Count flights by type
            dep_count = np.sum(window_flight_types == 'departure')
            arr_count = np.sum(window_flight_types == 'arrival')
            total_count = len(window_flight_types)
            
            results.append({
                'flight_id': flight_id,
                'airport_departures_60min': dep_count,
                'airport_arrivals_60min': arr_count,
                'airport_total_flights_60min': total_count,
                'flight_type': flight_type
            })
    
    print("Creating result dataframes...")
    congestion_df = pd.DataFrame(results)
    
    # Split by flight type
    dep_mask = congestion_df['flight_type'] == 'departure'
    arr_mask = congestion_df['flight_type'] == 'arrival'
    
    # Create departure features
    dep_features = congestion_df[dep_mask][['flight_id', 'airport_departures_60min', 
                                           'airport_arrivals_60min', 'airport_total_flights_60min']].copy()
    dep_features.columns = ['flight_id', 'dep_airport_departures_60min', 
                           'dep_airport_arrivals_60min', 'dep_airport_total_flights_60min']
    
    # Create arrival features
    arr_features = congestion_df[arr_mask][['flight_id', 'airport_departures_60min', 
                                           'airport_arrivals_60min', 'airport_total_flights_60min']].copy()
    arr_features.columns = ['flight_id', 'arr_airport_departures_60min', 
                           'arr_airport_arrivals_60min', 'arr_airport_total_flights_60min']
    
    # Merge with original data
    print("Merging with original data...")
    dep_congestion = df.merge(dep_features, on='flight_id', how='left')
    arr_congestion = df.merge(arr_features, on='flight_id', how='left')
    
    return dep_congestion, arr_congestion

print("Creating comprehensive airport congestion features...")
dep_congestion, arr_congestion = create_airport_congestion_features_fast(sched_data)

Creating congestion features...
Creating comprehensive airport congestion features...
Preparing data...
Processing 781712 flight events across 75 airports...
Processing 781712 flight events across 75 airports...
Creating result dataframes...
Merging with original data...


In [ ]:
print(f"Departure congestion shape: {dep_congestion.shape}")
print(f"Arrival congestion shape: {arr_congestion.shape}")

print("\n=== DEPARTURE CONGESTION FEATURES ===")
print("Sample departure congestion features:")
dep_sample = dep_congestion[['flight_id', 'dep_airport', 'std', 'dep_airport_departures_60min', 
                            'dep_airport_arrivals_60min', 'dep_airport_total_flights_60min']].head(10)
print(dep_sample)

print("\n=== ARRIVAL CONGESTION FEATURES ===")
print("Sample arrival congestion features:")
arr_sample = arr_congestion[['flight_id', 'arr_airport', 'sta', 'arr_airport_departures_60min', 
                            'arr_airport_arrivals_60min', 'arr_airport_total_flights_60min']].head(10)
print(arr_sample)

print("\n=== CONGESTION STATISTICS ===")
print(f"Max departure airport total flights: {dep_congestion['dep_airport_total_flights_60min'].max()}")
print(f"Average departure airport total flights: {dep_congestion['dep_airport_total_flights_60min'].mean():.2f}")
print(f"Max arrival airport total flights: {arr_congestion['arr_airport_total_flights_60min'].max()}")
print(f"Average arrival airport total flights: {arr_congestion['arr_airport_total_flights_60min'].mean():.2f}")

print("\n=== BUSIEST PERIODS ===")
print("Top 5 busiest departure periods:")
busy_dep = dep_congestion.nlargest(5, 'dep_airport_total_flights_60min')[
    ['dep_airport', 'std', 'dep_airport_departures_60min', 'dep_airport_arrivals_60min', 
     'dep_airport_total_flights_60min']
]
print(busy_dep)

Departure congestion shape: (390856, 7)
Arrival congestion shape: (390856, 7)

Sample departure congestion features:
       flight_id dep_airport                 std  dep_airport_congestion_60min
390851     WF955         VDS 2025-07-31 12:50:00                           1.0
390852    OU8448         VIE 2020-10-27 07:00:00                           1.0
390853   BIX9625         VXO 2023-12-28 15:00:00                           1.0
390854    LO7401         WAW 2025-03-05 08:45:00                           1.0
390855    GAV339         WMI 2024-11-08 12:30:00                           1.0


In [ ]:
# Analyze the congestion patterns
print("=== CONGESTION ANALYSIS ===")

# Look at airports with higher congestion
print("\nTop 10 busiest departure periods (highest congestion):")
busy_departures = dep_congestion.nlargest(10, 'dep_airport_congestion_60min')[['dep_airport', 'std', 'dep_airport_congestion_60min', 'dep_airport_is_peak_hour']]
print(busy_departures)

print("\nCongestion statistics:")
print(f"Max departure congestion: {dep_congestion['dep_airport_congestion_60min'].max()}")
print(f"Average departure congestion: {dep_congestion['dep_airport_congestion_60min'].mean():.2f}")
print(f"Flights during peak hours: {dep_congestion['dep_airport_is_peak_hour'].sum()} out of {len(dep_congestion)} ({dep_congestion['dep_airport_is_peak_hour'].mean()*100:.1f}%)")

print("\nAirports with most frequent high congestion (>5 flights in 60min):")
high_congestion = dep_congestion[dep_congestion['dep_airport_congestion_60min'] > 5]
if len(high_congestion) > 0:
    congestion_by_airport = high_congestion['dep_airport'].value_counts().head(10)
    print(congestion_by_airport)
else:
    print("No periods with >5 flights in 60 minutes found")

print("\n=== SAMPLE HIGH CONGESTION PERIODS ===")
sample_busy = dep_congestion[dep_congestion['dep_airport_congestion_60min'] > 3].head(10)
if len(sample_busy) > 0:
    print(sample_busy[['flight_id', 'dep_airport', 'std', 'dep_airport_congestion_60min', 'dep_airport_hour', 'dep_airport_is_peak_hour']])
else:
    print("No periods with >3 flights in 60 minutes found in sample")